# Pipeline 6: Campaign Timing and Seasonality Uplift

## When should fundraising campaigns run?

**Notebook:** `campaign-timing-seasonality.ipynb`  
**Domain:** Donor Strategy and Outreach  
**Purpose:** understand when the organization should schedule campaigns to maximize donation value and the chance of a high-value gift.

---

## 1. Problem framing

### Business question
The organization wants a clear answer to a practical fundraising question: which months, quarters, and campaign contexts are associated with stronger giving?

### Predictive and explanatory goals
- **Explanatory model:** estimate how calendar timing and campaign context relate to donation amount.
- **Predictive model:** predict whether a donation will land in the top-value quartile before the campaign is launched.

### Who uses this
- **Founders / leadership** to choose fundraising windows
- **Campaign planners** to compare seasons and channels
- **Volunteer marketers** to decide when to post or launch outreach

### Why this matters
A well-timed campaign can raise more money without increasing ad spend. This notebook turns historical donation timing into a decision aid.

---

## 2. Data and feature engineering

### Primary data source
- `donations`

### Features used
- Month, quarter, and year
- Year-end season flag
- Whether a campaign name is present
- Channel source
- Donation type
- Recurring vs one-time behavior

### Target variables
- **Regression target:** log donation amount
- **Classification target:** whether the donation is in the top quartile of value

### Leakage control
The notebook uses a clean time-aware split and avoids using any post-donation information as input features.

---

## 3. Modeling approach

### Explanatory track
A linear regression model estimates how calendar timing and campaign context are associated with donation value.

### Predictive track
A gradient boosting classifier predicts high-value donations, with a dummy baseline for comparison.

### Evaluation strategy
- 80/20 holdout split in time order
- 5-fold cross-validation on the classification target
- Permutation importance to identify the strongest timing signals

---

## 4. What the dashboard can show

### Useful insights
- Which quarter performs best
- Whether year-end months outperform the rest of the year
- Which campaign/channel combinations are most promising
- Whether recurring donors behave differently from one-time donors

### Decision output
The final output should help leadership plan the next campaign window and prioritize the channels most likely to produce larger gifts.

---

## 5. Caveats
This is observational data, so seasonality and campaign associations should be interpreted as patterns, not guaranteed causal effects.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import f1_score, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'ml-pipelines', Path.cwd().parent, Path.cwd().parent / 'ml-pipelines']:
    if (candidate / 'data_loader.py').exists():
        sys.path.insert(0, str(candidate))
        break
from data_loader import load_table

don = load_table('donations').sort_values('donation_date').copy()
don['amount_effective'] = don['amount'].fillna(don['estimated_value']).clip(lower=0)
don['month'] = don['donation_date'].dt.month
don['quarter'] = don['donation_date'].dt.quarter.astype(str)
don['year'] = don['donation_date'].dt.year
don['is_year_end'] = don['month'].isin([10, 11, 12]).astype(int)
don['campaign_present'] = don['campaign_name'].notna().astype(int)
don['target_log_amount'] = np.log1p(don['amount_effective'])
don['high_value'] = (don['amount_effective'] >= don['amount_effective'].quantile(0.75)).astype(int)

features = ['month', 'quarter', 'year', 'is_year_end', 'campaign_present', 'campaign_name', 'channel_source', 'is_recurring', 'donation_type']
X = don[features].copy()
y_reg = don['target_log_amount']
y_clf = don['high_value']

# Use explicit schema for stability across pandas dtype variations.
num_cols = ['month', 'year', 'is_year_end', 'campaign_present']
cat_cols = ['quarter', 'campaign_name', 'channel_source', 'is_recurring', 'donation_type']

# Normalize dtypes before sklearn transformers.
X[num_cols] = X[num_cols].apply(pd.to_numeric, errors='coerce')
for c in cat_cols:
    X[c] = X[c].astype('string').fillna('Unknown')

split = int(len(don) * 0.8)
Xtr, Xte = X.iloc[:split], X.iloc[split:]
ytr_reg, yte_reg = y_reg.iloc[:split], y_reg.iloc[split:]
ytr_clf, yte_clf = y_clf.iloc[:split], y_clf.iloc[split:]

prep = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

lin = Pipeline([('prep', prep), ('model', LinearRegression())])
lin.fit(Xtr, ytr_reg)
pred_reg = lin.predict(Xte)
print('Explanatory R2:', round(r2_score(yte_reg, pred_reg), 3))
print('Explanatory MAE:', round(mean_absolute_error(yte_reg, pred_reg), 3))

baseline = Pipeline([('prep', prep), ('model', DummyClassifier(strategy='prior'))])
gb = Pipeline([('prep', prep), ('model', GradientBoostingClassifier(random_state=42))])
baseline.fit(Xtr, ytr_clf)
gb.fit(Xtr, ytr_clf)
base_proba = baseline.predict_proba(Xte)[:, 1]
gb_proba = gb.predict_proba(Xte)[:, 1]
gb_pred = (gb_proba >= 0.5).astype(int)
print('Baseline AUC:', round(roc_auc_score(yte_clf, base_proba), 3))
print('GB AUC:', round(roc_auc_score(yte_clf, gb_proba), 3))
print('GB F1:', round(f1_score(yte_clf, gb_pred), 3))

cv = cross_validate(gb, X, y_clf, cv=5, scoring=['roc_auc', 'f1'])
print('CV AUC mean/std:', round(float(cv['test_roc_auc'].mean()), 3), round(float(cv['test_roc_auc'].std()), 3))

perm = permutation_importance(gb, Xte, yte_clf, n_repeats=8, random_state=42, scoring='roc_auc')
imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(10)
print('\nTop predictive drivers:')
print(imp.round(4).to_string())

coef_values = np.ravel(lin.named_steps['model'].coef_)
coef_names = lin.named_steps['prep'].get_feature_names_out()
usable = min(len(coef_values), len(coef_names))
coef = pd.Series(coef_values[:usable], index=coef_names[:usable]).sort_values(key=lambda s: s.abs(), ascending=False).head(10)
print('\nTop explanatory effects:')
print(coef.round(4).to_string())

print('\nDecision recommendations: prioritize Q4 and top-performing channel/campaign combinations for fundraising windows.')


Explanatory R2: 0.843
Explanatory MAE: 0.518
Baseline AUC: 0.5
GB AUC: 0.796
GB F1: 0.526
CV AUC mean/std: 0.72 0.064

Top predictive drivers:
donation_type       0.2033
month               0.0835
channel_source      0.0166
quarter             0.0028
campaign_present    0.0016
is_year_end         0.0010
year                0.0000
campaign_name      -0.0053
is_recurring       -0.0115

Top explanatory effects:
cat__donation_type_Monetary            2.6589
cat__donation_type_InKind              2.1225
cat__donation_type_SocialMedia        -2.0618
cat__donation_type_Skills             -1.5281
cat__donation_type_Time               -1.1915
num__month                             0.2527
cat__campaign_name_Year-End Hope      -0.2164
cat__quarter_1                         0.1823
cat__channel_source_PartnerReferral   -0.1293
cat__campaign_name_Summer of Safety    0.1255

Decision recommendations: prioritize Q4 and top-performing channel/campaign combinations for fundraising windows.


In [2]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'ml-pipelines', Path.cwd().parent, Path.cwd().parent / 'ml-pipelines']:
    if (candidate / 'trend_eval_helpers.py').exists():
        sys.path.insert(0, str(candidate))
        break
from trend_eval_helpers import print_calibration_bins, print_threshold_scan, bootstrap_linear_coefs, fairness_binary, fairness_regression_mae

print("\n=== Evaluation artifacts ===")
if yte_clf.nunique() >= 2:
    proba = gb.predict_proba(Xte)[:, 1]
    print_calibration_bins(yte_clf.values, proba)
    print_threshold_scan(yte_clf.values, proba)
    fairness_binary(gb, Xte, yte_clf, don.loc[Xte.index], ['quarter'], min_n=20)
bootstrap_linear_coefs(lin, Xtr, ytr_reg, n_boot=150, top_k=8)



=== Evaluation artifacts ===
  bin [0.00,0.10): n=34 mean_pred=0.012 rate_pos=0.059
  bin [0.10,0.20): n=2 mean_pred=0.186 rate_pos=0.000
  bin [0.20,0.30): n=7 mean_pred=0.244 rate_pos=0.000
  bin [0.30,0.40): n=12 mean_pred=0.368 rate_pos=0.583
  bin [0.40,0.50): n=12 mean_pred=0.456 rate_pos=0.167
  bin [0.50,0.60): n=8 mean_pred=0.552 rate_pos=0.625
  bin [0.60,0.70): n=5 mean_pred=0.662 rate_pos=0.600
  bin [0.70,0.80): n=4 mean_pred=0.733 rate_pos=0.500
  thr  prec   rec    F1
  0.1  0.380  0.905  0.535
  0.2  0.396  0.905  0.551
  0.3  0.463  0.905  0.613
  0.4  0.414  0.571  0.480
  0.5  0.588  0.476  0.526
  0.6  0.556  0.238  0.333
  0.7  0.500  0.095  0.160
  0.8  0.000  0.000  0.000
  0.9  0.000  0.000  0.000

--- Fairness-style slices (AUC), min_n=20 ---
  quarter=3: n=17 (skip n<min_n)
  quarter=4: n=48 AUC=0.729
  quarter=1: n=19 (skip n<min_n)
  feature  median  p2.5  p97.5
  cat__donation_type_Monetary                       +2.6575  +2.5593  +2.7563
  cat__donation_ty